# 02 - Ti?n X? L? & Feature Engineering

**M?c ti?u:**
- X? l? missing values
- X? l? outlier
- Encoding bi?n ph?n lo?i
- R?i r?c ho? cho Apriori
- L?u d? li?u ?? x? l?

In [ ]:
import sys
sys.path.insert(0, '..')
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 2.1 Load d? li?u th?

In [ ]:
from src.data.loader import load_raw_data, load_params

params = load_params()
df = load_raw_data()
print(f"Shape ban ??u: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")

## 2.2 Pipeline ti?n x? l?

C?c b??c:
1. Drop c?t kh?ng c?n (`id`, `dataset`)
2. T?o binary target (num ? target)
3. X? l? missing values (fill median/mode)
4. Encode bi?n ph?n lo?i (Label Encoding)
5. Handle outlier (IQR clipping)

In [ ]:
from src.data.cleaner import full_cleaning_pipeline

df_clean = full_cleaning_pipeline(
    df,
    drop_cols=params['preprocessing']['drop_cols'],
    missing_strategy=params['preprocessing']['missing_strategy'],
    missing_threshold=params['preprocessing']['missing_threshold'],
    encode_method=params['preprocessing']['encode_method'],
    handle_outliers=params['preprocessing']['handle_outliers'],
)

In [ ]:
# Ki?m tra k?t qu?
print(f"Shape sau ti?n x? l?: {df_clean.shape}")
print(f"Missing values: {df_clean.isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(df_clean['target'].value_counts())
df_clean.head()

## 2.3 Ki?m tra Duplicates

In [ ]:
# Ki?m tra duplicate
dupes = df_clean.duplicated().sum()
print(f"S? d?ng tr?ng l?p: {dupes}")
if dupes > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"?? xo? {dupes} d?ng. Shape m?i: {df_clean.shape}")
else:
    print("Kh?ng c? d?ng tr?ng l?p")

## 2.4 Th?ng k? tr??c vs sau ti?n x? l?

In [ ]:
# So s?nh tr??c - sau chi ti?t
comparison = pd.DataFrame({
    'Before': [df.shape[0], df.shape[1], df.isnull().sum().sum(), df.duplicated().sum()],
    'After': [df_clean.shape[0], df_clean.shape[1], df_clean.isnull().sum().sum(), df_clean.duplicated().sum()]
}, index=['Rows', 'Columns', 'Missing Values', 'Duplicates'])
print("=== TH?NG K? TR??C vs SAU TI?N X? L? ===")
comparison

In [ ]:
# So s?nh mean/std tr??c-sau cho bi?n s?
before_stats = df.select_dtypes(include='number').describe().loc[['mean','std']].T
after_stats = df_clean.select_dtypes(include='number').describe().loc[['mean','std']].T
compare_stats = pd.DataFrame({
    'Before Mean': before_stats['mean'],
    'After Mean': after_stats['mean'],
    'Before Std': before_stats['std'],
    'After Std': after_stats['std']
}).dropna().round(2)
print("=== SO S?NH MEAN/STD TR??C-SAU ===")
compare_stats

In [ ]:
# Ph?n ph?i c?c bi?n s? sau ti?n x? l?
from src.visualization.plots import plot_numeric_distributions
fig = plot_numeric_distributions(df_clean, save=False)
plt.suptitle('Distributions After Preprocessing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2.4 Feature Engineering - R?i r?c ho? cho Apriori

In [ ]:
from src.features.builder import discretize_for_apriori

# D?ng d? li?u g?c (ch?a encode) cho r?i r?c ho?
df_for_disc = df.drop(columns=['id', 'dataset']).copy()
df_for_disc['target'] = (df_for_disc['num'] > 0).astype(int)

df_disc = discretize_for_apriori(df_for_disc)
print(f"\nShape: {df_disc.shape}")
df_disc.head()

## 2.5 Chu?n b? X, y cho Modeling

In [ ]:
from src.features.builder import select_features_for_modeling

X, y = select_features_for_modeling(df_clean)

## 2.6 L?u d? li?u ?? x? l?

In [ ]:
import os
os.makedirs(os.path.dirname(params['paths']['processed_data']), exist_ok=True)

# L?u processed data
df_clean.to_csv(params['paths']['processed_data'], index=False)
print(f"?? Saved: {params['paths']['processed_data']}")

# L?u discretized data
df_disc.to_csv(params['paths']['discretized_data'], index=False)
print(f"?? Saved: {params['paths']['discretized_data']}")

## 2.7 Nh?n x?t

- **Missing values:** X? l? b?ng median (s?) v? mode (ph?n lo?i)
- **Encoding:** Label Encoding cho 5 bi?n ph?n lo?i
- **Outlier:** Ch? clip 4 bi?n li?n t?c th?t s? (age, trestbps, chol, thalch, oldpeak)
- **R?i r?c ho?:** T?o ???c binary features cho Apriori d?a tr?n ng??ng y t?
- D? li?u s?n s?ng cho b??c Mining v? Modeling